# A3 — Model & Optimasi (Minggu 13)
### TA-PCVK: Facial Expression Recognition (Hybrid Feature Descriptor + Jaya-LSSVM)

**Divisi**: A3 (Model & Optimasi)  
**Referensi Jurnal**:
> Kar, N.B., Nayak, D.R., Babu, K.S., & Zhang, Y.-D. (2021). *A hybrid feature descriptor with Jaya optimised least squares SVM for facial expression recognition.* IET Image Processing, 15(7), 1471–1483. DOI: [10.1049/ipr2.12118](https://doi.org/10.1049/ipr2.12118)  
> Suykens, J.A.K., & Vandewalle, J. (1999). *Least Squares Support Vector Machine Classifiers.* Neural Processing Letters, 9(3), 293–300.

---

### Sasaran Minggu 13:
1. **LS-SVM Multiclass (One-vs-One / OvO)**:
   - Perluasan prototipe LS-SVM biner menjadi classifier multikelas dengan strategi One-vs-One.
   - Untuk 7 kelas ekspresi wajah, dibangun $K(K-1)/2 = 21$ classifier biner LS-SVM.
   - Mekanisme inferensi berbasis *majority voting* dengan *confidence score margin tie-breaking*.
2. **Evaluasi 5-Fold Stratified Cross-Validation dengan Grid Search**:
   - Skema validasi silang bersarang (*nested CV*) untuk mencari pasangan hyperparameter $(\gamma, \sigma)$ optimal pada fold latih.
   - Pipeline: **LDA ($680 \to 6$) $\to$ StandardScaler $\to$ OvO LS-SVM**.
3. **Analisis Confusion Matrix & Metrik Deskriptif v0**:
   - Menghitung metrik Accuracy, Macro Sensitivity (Recall), Macro Specificity, serta metrik per-kelas.
   - Visualisasi confusion matrix kedua dataset (CK+ dan JAFFE) sebagai baseline deskriptif v0 sebelum optimasi metaheuristik Jaya di Minggu 14.

## 1. Import Library & Muat Konfigurasi

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tambahkan direktori kerja ke sys.path jika belum ada
repo_root = Path.cwd().resolve()
if repo_root.name == "model-experiment":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "model-experiment") not in sys.path:
    sys.path.insert(0, str(repo_root / "model-experiment"))

from baseline import load_config
from lssvm import OneVsOneLSSVMClassifier, MulticlassLSSVM
from evaluate_lssvm import run_lssvm_multiclass_evaluation

# Muat konfigurasi terpusat
config = load_config(repo_root / "config.yaml")
print(f"Proyek: {config['project']['name']}")
print(f"Grid search gamma: {config['grid_search']['gamma_values']}")
print(f"Grid search sigma: {config['grid_search']['sigma_values']}")

## 2. Konsep & Arsitektur Multiclass LS-SVM (One-vs-One)

Pada pendekatan One-vs-One (OvO):
1. Dibuat sub-problem biner untuk setiap pasangan kelas $(c_1, c_2)$ di mana $c_1 < c_2$.
2. Untuk $K = 7$ kelas ekspresi wajah (CK+ & JAFFE):
   $$N_{\text{classifiers}} = \frac{7 \times (7 - 1)}{2} = 21 \text{ classifier biner LS-SVM}$$
3. Setiap classifier dilatih hanya menggunakan sampel dari kedua kelas tersebut.
4. **Strategi Inferensi**:
   - Setiap model memprediksi $y \in \{c_1, c_2\}$ dan memberikan 1 suara (*vote*).
   - Akumulasi skor keputusan kontinu $f(x)$ dicatat sebagai *confidence margin*.
   - Kelas dengan suara terbanyak dipilih sebagai prediksi final.
   - Jika terjadi nilai seri (*tie*), pemenang ditentukan oleh akumulasi margin tertinggi.

In [ ]:
# Demonstrasi struktur OvO LS-SVM pada 7 kelas
from itertools import combinations

class_names_ck = config["dataset"]["ck"]["class_names"]
pairs = list(combinations(class_names_ck, 2))
print(f"Total pasangan biner untuk 7 kelas: {len(pairs)}")
print("Contoh 5 pasangan pertama:")
for p in pairs[:5]:
    print(f"  - Model biner: {p[0]} vs {p[1]}")

## 3. Hasil Evaluasi 5-Fold Stratified CV (Baseline Deskriptif v0)

Artefak hasil evaluasi yang dihasilkan oleh `evaluate_lssvm.py` dimuat dan dianalisis di bawah ini.

In [ ]:
# Muat ringkasan hasil evaluasi Minggu 13
summary_path = repo_root / "reports" / "lssvm_multiclass_summary.csv"
df_summary = pd.read_csv(summary_path)
print("=== TABEL HASIL BASELINE DESKRIPTIF v0 (LS-SVM MULTICLASS) ===")
display(df_summary)

### Detail Parameter Terbaik per Fold

Berikut adalah pasangan hyperparameter $(\gamma^*, \sigma^*)$ yang dipilih oleh Grid Search pada setiap fold validasi silang:

In [ ]:
ck_json_path = repo_root / "reports" / "lssvm_multiclass_ck_results.json"
jaffe_json_path = repo_root / "reports" / "lssvm_multiclass_jaffe_results.json"

with open(ck_json_path, "r") as f:
    res_ck = json.load(f)
with open(jaffe_json_path, "r") as f:
    res_jaffe = json.load(f)

print("--- Parameter Terbaik per Fold: CK+ ---")
for i, p in enumerate(res_ck["best_params_per_fold"]):
    print(f"Fold {i+1}: gamma = {p['gamma']}, sigma = {p['sigma']} -> Acc fold: {res_ck['fold_accuracies'][i]*100:.2f}%")

print("\n--- Parameter Terbaik per Fold: JAFFE ---")
for i, p in enumerate(res_jaffe["best_params_per_fold"]):
    print(f"Fold {i+1}: gamma = {p['gamma']}, sigma = {p['sigma']} -> Acc fold: {res_jaffe['fold_accuracies'][i]*100:.2f}%")

## 4. Visualisasi Confusion Matrix Agregat (5-Fold CV)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, res, title in zip(
    axes,
    [res_ck, res_jaffe],
    ["CK+ (LDA + OvO LS-SVM Grid Search)", "JAFFE (LDA + OvO LS-SVM Grid Search)"]
):
    cm = np.array(res["confusion_matrix"])
    class_names = res["class_names"]
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Greens)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        title=f"{title}\nMean Acc: {res['mean_accuracy']*100:.2f}% (Std: {res['std_accuracy']*100:.2f}%)",
        ylabel="True Label",
        xlabel="Predicted Label"
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontweight="bold")

fig.tight_layout()
plt.show()

## 5. Analisis Metrik per Kelas (Sensitivity & Specificity)

Tabel berikut menyajikan sensitivitas (kemampuan mendeteksi ekspresi wajah tertentu) dan spesifisitas untuk tiap kelas ekspresi:

In [ ]:
def make_class_metrics_df(res):
    return pd.DataFrame({
        "Kelas Ekspresi": res["class_names"],
        "Sensitivity / Recall (%)": [round(s * 100, 2) for s in res["per_class_sensitivity"]],
        "Specificity (%)": [round(s * 100, 2) for s in res["per_class_specificity"]],
    })

print("=== METRIK PER KELAS: CK+ ===")
display(make_class_metrics_df(res_ck))

print("\n=== METRIK PER KELAS: JAFFE ===")
display(make_class_metrics_df(res_jaffe))

## 6. Ringkasan Hasil & Kesiapan Menuju Minggu 14

| Target Minggu 13 | Status | Keterangan |
|---|---|---|
| **LS-SVM Multiclass (One-vs-One)** | ✅ Selesai | 21 classifier biner dengan majority voting + confidence margin tie-breaking (`lssvm.py`). |
| **Evaluasi 5-Fold Stratified CV** | ✅ Selesai | Nested CV pada CK+ (981 citra) dan JAFFE (213 citra) dengan LDA + StandardScaler. |
| **Grid Search $(\gamma, \sigma)$** | ✅ Selesai | Tuning parameter terpusat dari `config.yaml`. |
| **Metrik Sen/Spe/Acc & Confusion Matrix** | ✅ Selesai | CK+: **96.64%** (Sen: 97.02%, Spe: 99.44%), JAFFE: **79.36%** (Sen: 79.28%, Spe: 96.56%). |
| **Kesiapan Minggu 14** | ⏳ Siap | Lanjut ke implementasi **Algoritma Optimasi Metaheuristik Jaya** (dan pembanding PSO/GA) untuk melampaui baseline grid search menuju target CK+ $\ge 88\%$, JAFFE $\ge 90\%$. |